# Read YOLO Dataset Annotations to DataFrame

This notebook reads all YOLO annotations from train, val, holdout, and holdout_v2 splits,
extracts camera height from filenames, and saves to CSV.

In [13]:
import os
import re
import pandas as pd
from pathlib import Path
from tqdm import tqdm

In [14]:
# Dataset path
DATASET_PATH = "/home/klymy/data/cam4_revB/yolo_dataset/batch1_10_margin_15_bg_wb_alif_clahe/labels"
OUTPUT_CSV = "yolo_annotations.csv"

# Subsets to process
SUBSETS = ["train", "val", "holdout", "holdout_v2"]

In [15]:
def extract_camera_height(filename: str) -> int | None:
    """
    Extract camera height from filename and round to nearest 10 cm.

    Examples:
    - 4.2_batch1____4.2_H260_menivim_5.txt -> 260
    - 4.2_batch1____4.2_H277_menivim_OS_28.txt -> 280
    - 4.2_batch2____4.2_H_cam_234_spacesbooth_100.txt -> 230
    - 4.2_H289_file.txt -> 290
    - 4.2_H272_file.txt -> 270

    Args:
        filename: Name of the annotation file

    Returns:
        Camera height in cm (rounded to nearest 10), or None if not found
    """
    # Try pattern: H<number> or H_cam_<number>
    patterns = [
        r'H_cam_(\d+)',  # H_cam_234
        r'H(\d+)',        # H260
    ]

    for pattern in patterns:
        match = re.search(pattern, filename)
        if match:
            height = int(match.group(1))
            # Round to nearest 10 cm
            return round(height / 10) * 10

    return None

In [16]:
def read_yolo_annotations(labels_dir: str, subsets: list[str]) -> pd.DataFrame:
    """
    Read YOLO annotations from multiple subsets and create a DataFrame.
    
    Args:
        labels_dir: Path to labels directory
        subsets: List of subset names (train, val, holdout, holdout_v2)
    
    Returns:
        DataFrame with columns: filename, x, y, w, h, pos_x, pos_y, cam_height
    """
    all_annotations = []
    skipped_files = []
    
    for subset in subsets:
        subset_path = os.path.join(labels_dir, subset)
        
        if not os.path.exists(subset_path):
            print(f"Warning: {subset_path} does not exist, skipping...")
            continue
        
        txt_files = list(Path(subset_path).glob("*.txt"))
        print(f"Processing {subset}: {len(txt_files)} files")
        
        for txt_file in tqdm(txt_files, desc=f"Reading {subset}"):
            filename = txt_file.name
            
            # Extract camera height
            cam_height = extract_camera_height(filename)
            
            if cam_height is None:
                skipped_files.append(str(txt_file))
                continue
            
            # Read YOLO annotations from file
            try:
                with open(txt_file, 'r') as f:
                    for line in f:
                        line = line.strip()
                        if not line:
                            continue
                        
                        # YOLO format: class_id x_center y_center width height
                        parts = line.split()
                        if len(parts) < 5:
                            continue
                        
                        class_id, x, y, w, h = parts[:5]
                        x, y, w, h = float(x), float(y), float(w), float(h)
                        
                        # Calculate position (top-left corner)
                        pos_x = x - w / 2
                        pos_y = y - h / 2
                        
                        all_annotations.append({
                            'filename': filename,
                            'x': x,
                            'y': y,
                            'w': w,
                            'h': h,
                            'pos_x': pos_x,
                            'pos_y': pos_y,
                            'cam_height': cam_height,
                            'subset': subset
                        })
            except Exception as e:
                print(f"Error reading {txt_file}: {e}")
    
    print(f"\nSkipped {len(skipped_files)} files without camera height information")
    
    return pd.DataFrame(all_annotations)

In [17]:
# Test camera height extraction and rounding
test_filenames = [
    "4.2_batch1____4.2_H260_menivim_5.txt",
    "4.2_batch1____4.2_H277_menivim_OS_28.txt",
    "4.2_batch2____4.2_H_cam_234_spacesbooth_100.txt",
    "4.2_background____4.2_H_cam_180__5.txt",
    "4.2_H289_test.txt",
    "4.2_H272_test.txt"
]

print("Testing camera height extraction (with rounding to nearest 10 cm):")
for filename in test_filenames:
    height = extract_camera_height(filename)
    print(f"{filename} -> {height} cm")

Testing camera height extraction (with rounding to nearest 10 cm):
4.2_batch1____4.2_H260_menivim_5.txt -> 260 cm
4.2_batch1____4.2_H277_menivim_OS_28.txt -> 280 cm
4.2_batch2____4.2_H_cam_234_spacesbooth_100.txt -> 230 cm
4.2_background____4.2_H_cam_180__5.txt -> 180 cm
4.2_H289_test.txt -> 290 cm
4.2_H272_test.txt -> 270 cm


In [18]:
# Read all annotations
df = read_yolo_annotations(DATASET_PATH, SUBSETS)

print(f"\nTotal annotations: {len(df)}")
print(f"Unique files: {df['filename'].nunique()}")
print(f"\nDataset splits:")
print(df['subset'].value_counts())

Processing train: 6723 files


Reading train:  15%|█▌        | 1010/6723 [00:00<00:00, 10093.61it/s]

Reading train: 100%|██████████| 6723/6723 [00:00<00:00, 14272.43it/s]


Processing val: 742 files


Reading val: 100%|██████████| 742/742 [00:00<00:00, 15884.43it/s]


Processing holdout: 148 files


Reading holdout: 100%|██████████| 148/148 [00:00<00:00, 14618.43it/s]


Processing holdout_v2: 191 files


Reading holdout_v2: 100%|██████████| 191/191 [00:00<00:00, 15233.16it/s]



Skipped 4 files without camera height information

Total annotations: 31766
Unique files: 7516

Dataset splits:
subset
train         27644
val            3106
holdout_v2      609
holdout         407
Name: count, dtype: int64


In [19]:
# Display sample data
df.head(10)

,filename,x,y,w,h,pos_x,pos_y,cam_height,subset
0,4.2_batch7____4.2_H_cam_289_menivimOS_31.txt,0.669141,0.578896,0.171094,0.219708,0.583594,0.469042,290,train
1,4.2_batch7____4.2_H_cam_289_menivimOS_31.txt,0.349453,0.726865,0.166719,0.220396,0.266094,0.616667,290,train
2,4.2_batch6____4.2_H_cam_330_spacesOS_136.txt,0.924188,0.329885,0.151625,0.146021,0.848375,0.256875,330,train
3,4.2_batch6____4.2_H_cam_330_spacesOS_136.txt,0.762734,0.706562,0.120156,0.196458,0.702656,0.608333,330,train
4,4.2_batch6____4.2_H_cam_330_spacesOS_110.txt,0.367031,0.320104,0.193125,0.212708,0.270469,0.213750,330,train
5,4.2_batch6____4.2_H_cam_330_spacesOS_110.txt,0.720859,0.710729,0.115781,0.257708,0.662969,0.581875,330,train
6,4.2_batch6____4.2_H_cam_330_spacesOS_110.txt,0.550656,0.726135,0.136500,0.175604,0.482406,0.638333,330,train
7,4.2_batch6____4.2_H_cam_330_spacesOS_110.txt,0.916336,0.352396,0.162359,0.129375,0.835156,0.287708,330,train
8,4.2_batch10____4.2_H_cam_229_Kitchen_397.txt,0.855406,0.341177,0.220719,0.191271,0.745047,0.245542,230,train
9,4.2_batch10____4.2_H_cam_229_Kitchen_397.txt,0.868992,0.082885,0.107734,0.165771,0.815125,0.000000,230,train


In [20]:
# Check camera height distribution
print("Camera height distribution:")
print(df['cam_height'].value_counts().sort_index())

Camera height distribution:
cam_height
20       65
80      449
220      10
230    3607
240    1740
250    9858
260    5625
270     317
280    2636
290    1864
300    2249
320    2372
330     810
340     164
Name: count, dtype: int64


In [21]:
df = df[df['cam_height'] >= 200]

In [22]:
# Display DataFrame info
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 31252 entries, 0 to 31765
Data columns (total 9 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   filename    31252 non-null  object 
 1   x           31252 non-null  float64
 2   y           31252 non-null  float64
 3   w           31252 non-null  float64
 4   h           31252 non-null  float64
 5   pos_x       31252 non-null  float64
 6   pos_y       31252 non-null  float64
 7   cam_height  31252 non-null  int64  
 8   subset      31252 non-null  object 
dtypes: float64(6), int64(1), object(2)
memory usage: 2.4+ MB


In [23]:
# Save to CSV
df.to_csv(OUTPUT_CSV, index=False)
print(f"\nAnnotations saved to {OUTPUT_CSV}")
print(f"File size: {os.path.getsize(OUTPUT_CSV) / 1024 / 1024:.2f} MB")


Annotations saved to yolo_annotations.csv
File size: 4.60 MB


In [24]:
# Summary statistics
print("\n=== Summary Statistics ===")
print(f"Total bounding boxes: {len(df)}")
print(f"\nBy subset:")
for subset in SUBSETS:
    count = len(df[df['subset'] == subset])
    files = df[df['subset'] == subset]['filename'].nunique()
    print(f"  {subset}: {count} boxes in {files} files")

print(f"\nCamera heights (cm):")
print(f"  Min: {df['cam_height'].min()}")
print(f"  Max: {df['cam_height'].max()}")
print(f"  Mean: {df['cam_height'].mean():.1f}")
print(f"  Unique values: {df['cam_height'].nunique()}")


=== Summary Statistics ===
Total bounding boxes: 31252

By subset:
  train: 27192 boxes in 6408 files
  val: 3044 boxes in 709 files
  holdout: 407 boxes in 141 files
  holdout_v2: 609 boxes in 184 files

Camera heights (cm):
  Min: 220
  Max: 340
  Mean: 265.5
  Unique values: 12
